# SHIVA-LLM — Objective 02, Stage 2: Frozen HyperKT State Export
**Runtime:** Google Colab **T4 GPU**, Python 3. CPU fallback supported (slower). **No retraining.**

**Required inputs:** (1) Objective 01 **Stage 2 ZIP** containing `train_sequences.npz`, `val_sequences.npz`, `test_sequences.npz`, `stage2_report.json`; (2) **Stage 3B ZIP** containing `hypergraph_repaired.npz`; (3) **Stage 9E checkpoint ZIP** with `HyperKT_gated_seed42/best.pt` (or the original Stage 9E Drive directory). The user-uploaded Stage 9E full archive is suitable. **Stage 9–11 result-only ZIPs do not replace Stage 2/3B preprocessing artifacts.**

**Research safeguard:** This notebook exports *frozen, pre-response predictions* for the next skill from causal history. These are one-dimensional knowledge estimates, **not the complete latent embeddings**. Sequence-window rows are indexed with `(split, window_index, position)`. Do **not** join directly to Objective 02 Stage 1 transitions on `user_id` alone: source sequence-event identity must first be verified. Held-out test predictions must never be used for policy training, reward fitting, or tuning.


In [ ]:
%pip -q install numpy pandas torch nbformat

In [ ]:
import io, os, json, zipfile, hashlib, pathlib, random
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from google.colab import files,drive
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Runtime:',DEVICE,torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
drive.mount('/content/drive')
OUT=Path('/content/drive/MyDrive/SHIVA_LLM_Objective02/Stage2');OUT.mkdir(parents=True,exist_ok=True)


## Upload the two original preprocessing ZIPs and the Stage 9E full checkpoint ZIP
You can upload them from your computer or copy their paths into `INPUT_PATHS` if already stored in Drive. Upload **three files**; archives are inspected by their actual contents, not by filenames.

In [ ]:
INPUT_PATHS=[]  # e.g. ['/content/drive/MyDrive/HyperKT/stage2.zip', ...]
if not INPUT_PATHS:
    uploaded=files.upload()
    INPUT_PATHS=list(uploaded)
assert len(INPUT_PATHS)>=3,'Need Stage 2, Stage 3B, and Stage 9E checkpoint ZIPs.'
zs={str(p):zipfile.ZipFile(str(p)) for p in INPUT_PATHS}
def member(z,suffix):
    matches=[x for x in z.namelist() if x.endswith(suffix) and not x.startswith('__MACOSX')]
    if len(matches)!=1: raise ValueError(f'Expected exactly one {suffix}, found {matches[:4]}')
    return matches[0]
def identify(z):
    names=z.namelist()
    if any(x.endswith('train_sequences.npz') for x in names):return 'stage2'
    if any(x.endswith('hypergraph_repaired.npz') for x in names):return 'stage3b'
    if any(x.endswith('HyperKT_gated_seed42/best.pt') for x in names):return 'stage9e'
    return 'unknown'
bytype={}
for p,z in zs.items():
    t=identify(z);print(p,'=>',t)
    assert t!='unknown' and t not in bytype,'Unknown/duplicate archive type'
    bytype[t]=z
assert set(bytype)=={'stage2','stage3b','stage9e'},'Missing one of the three required archive types'
z2,z3,z9=(bytype[x] for x in ['stage2','stage3b','stage9e'])
def read_npz(z,name):
    with np.load(io.BytesIO(z.read(member(z,name))),allow_pickle=False) as d:
        return {k:d[k].copy() for k in d.files}
train=read_npz(z2,'train_sequences.npz');val=read_npz(z2,'val_sequences.npz');test=read_npz(z2,'test_sequences.npz')
struct=read_npz(z3,'hypergraph_repaired.npz')
manifest=json.loads(z2.read(member(z2,'stage2_report.json')))
N_SKILLS=int(max(train['target_skill'].max(),val['target_skill'].max(),test['target_skill'].max(),struct['skill_ids'].max())+1)
H=np.zeros((N_SKILLS,struct['assignments'].shape[1]),dtype=np.float32)
H[struct['skill_ids'].astype(int)]=struct['assignments'].astype(np.float32)
MAX_LEN=train['history_skill'].shape[1]
print('skills',N_SKILLS,'sequence length',MAX_LEN,'hyperedges',H.shape[1]);print('manifest keys:',list(manifest))
assert N_SKILLS==151 and MAX_LEN==200,'Input configuration differs from frozen Stage 9E experiment'
for split,d in [('train',train),('val',val),('test',test)]:
    print(split,'windows',len(d['mask']),'events',int(d['mask'].sum()),'keys',list(d))
assert int(train['mask'].sum())==222562 and int(val['mask'].sum())==23858 and int(test['mask'].sum())==32522


## Reconstruct the exact Stage 9E architecture
The following definitions are copied from the uploaded Stage 9E training notebook. They are not an approximation.

In [ ]:
VARIANTS=["DKT_matched","HyperKT_matched","HyperKT_gated"]
class SeqDataset(Dataset):
    def __init__(self,d): self.d=d
    def __len__(self):return len(self.d['mask'])
    def __getitem__(self,i):
        return tuple(torch.as_tensor(self.d[k][i],dtype=torch.long if k in ('history_skill','history_correct','target_skill') else torch.float32) for k in ('history_skill','history_correct','target_skill','target_correct','mask'))
TRAIN_DS=SeqDataset(train); VAL_DS=SeqDataset(val)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
    try:torch.use_deterministic_algorithms(True,warn_only=True)
    except Exception:pass
class CausalBlock(nn.Module):
    def __init__(self,d=128,heads=8,drop=.1,decay=False):
        super().__init__();self.heads=heads;self.dhead=d//heads;self.decay=decay
        self.qkv=nn.Linear(d,3*d);self.proj=nn.Linear(d,d);self.drop=nn.Dropout(drop)
        self.ln1=nn.LayerNorm(d);self.ln2=nn.LayerNorm(d)
        self.ff=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Dropout(drop),nn.Linear(4*d,d))
        if decay:self.log_rate=nn.Parameter(torch.full((heads,),-3.0))
    def forward(self,x):
        B,T,D=x.shape; z=self.ln1(x)
        q,k,v=self.qkv(z).chunk(3,dim=-1)
        def split(t):return t.reshape(B,T,self.heads,self.dhead).transpose(1,2)
        q,k,v=map(split,(q,k,v))
        logits=torch.matmul(q,k.transpose(-2,-1))*(self.dhead**-.5)
        pos=torch.arange(T,device=x.device)
        dist=(pos[:,None]-pos[None,:]).clamp(min=0).float()
        if self.decay:logits=logits-torch.nn.functional.softplus(self.log_rate)[None,:,None,None]*dist[None,None,:,:]
        future=pos[None,:]>pos[:,None]
        logits=logits.masked_fill(future[None,None,:,:],torch.finfo(logits.dtype).min)
        a=torch.softmax(logits,dim=-1)
        x=x+self.drop(self.proj(torch.matmul(self.drop(a),v).transpose(1,2).reshape(B,T,D)))
        x=x+self.drop(self.ff(self.ln2(x)))
        return x
class HyperKT(nn.Module):
    def __init__(self,n_skills,H,hyper=False,decay=False,d=128,layers=2,heads=8,drop=.1):
        super().__init__();self.hyper=hyper
        self.skill=nn.Embedding(n_skills,d,padding_idx=1)
        self.response=nn.Embedding(2,d)
        self.pos=nn.Embedding(MAX_LEN,d)
        self.blocks=nn.ModuleList([CausalBlock(d,heads,drop,decay) for _ in range(layers)])
        self.norm=nn.LayerNorm(d)
        if hyper:
            self.register_buffer('assignments',torch.tensor(H))
            self.edge_embedding=nn.Parameter(torch.randn(H.shape[1],d)*.02)
            self.fusion=nn.Linear(d*2,d)
        self.output=nn.Sequential(nn.LayerNorm(d*2),nn.Linear(d*2,d),nn.GELU(),nn.Dropout(drop),nn.Linear(d,1))
    def forward(self,hs,hc,ts):
        B,T=hs.shape; pos=torch.arange(T,device=hs.device)[None,:]
        x=self.skill(hs)+self.response(hc.clamp(0,1))+self.pos(pos)
        for layer in self.blocks:x=layer(x)
        x=self.norm(x)
        target=self.skill(ts)
        if self.hyper:
            structural=self.assignments[ts]@self.edge_embedding
            target=target+torch.tanh(self.fusion(torch.cat([target,structural],dim=-1)))
        return self.output(torch.cat([x,target],dim=-1)).squeeze(-1)
VARIANTS={'DKT_matched':None,'HyperKT_matched':None,'HyperKT_gated':None}
SEEDS=[42,123,2024,3407,7777]

class DKT(nn.Module):
    # Independent Stage 9C baseline implementation, not a certified official DKT reproduction.
    def __init__(self,n,d=128,dropout=.1):
        super().__init__();self.n=n;self.emb=nn.Embedding(n*2,d)
        self.rnn=nn.LSTM(d,d,batch_first=True);self.drop=nn.Dropout(dropout);self.out=nn.Linear(d,n)
    def forward(self,hs,hc,ts):
        h,_=self.rnn(self.emb(hs+hc.clamp(0,1)*self.n))
        return self.out(self.drop(h)).gather(-1,ts.unsqueeze(-1)).squeeze(-1)

class HyperKTGated(HyperKT):
    # Only change vs original HyperKT: a scalar learned gate on the hypergraph residual.
    def __init__(self,n_skills,H,d=128,layers=2,heads=8,drop=.1):
        super().__init__(n_skills,H,hyper=True,decay=True,d=d,layers=layers,heads=heads,drop=drop)
        self.hyper_gate=nn.Parameter(torch.tensor(-2.0))
    def forward(self,hs,hc,ts):
        B,T=hs.shape;pos=torch.arange(T,device=hs.device)[None,:]
        x=self.skill(hs)+self.response(hc.clamp(0,1))+self.pos(pos)
        for layer in self.blocks:x=layer(x)
        x=self.norm(x);target=self.skill(ts)
        structural=self.assignments[ts]@self.edge_embedding
        target=target+torch.sigmoid(self.hyper_gate)*torch.tanh(self.fusion(torch.cat([target,structural],dim=-1)))
        return self.output(torch.cat([x,target],dim=-1)).squeeze(-1)

def make_model(name):
    if name=='DKT_matched':return DKT(N_SKILLS).to(DEVICE)
    if name=='HyperKT_matched':return HyperKT(N_SKILLS,H,hyper=True,decay=True).to(DEVICE)
    if name=='HyperKT_gated':return HyperKTGated(N_SKILLS,H).to(DEVICE)
    raise ValueError(name)

# No future responses or target skills should influence earlier predictions.
seed_all(42)
for name in VARIANTS:
    m=make_model(name).eval()
    hs=torch.as_tensor(val['history_skill'][:2],device=DEVICE,dtype=torch.long)
    hc=torch.as_tensor(val['history_correct'][:2],device=DEVICE,dtype=torch.long)
    ts=torch.as_tensor(val['target_skill'][:2],device=DEVICE,dtype=torch.long)
    with torch.no_grad():
        base=m(hs,hc,ts)
        changed_hc=hc.clone();changed_ts=ts.clone();changed_hc[:,100:]=1-changed_hc[:,100:]
        changed_ts[:,100:]=(changed_ts[:,100:]+2)%N_SKILLS
        altered=m(hs,changed_hc,changed_ts)
        delta=(base[:,:100]-altered[:,:100]).abs().max().item()
    assert delta<1e-5,(name,delta)
    print(name,'causal max difference',delta,'parameters',sum(p.numel() for p in m.parameters()))
    del m
if torch.cuda.is_available():torch.cuda.empty_cache()

In [ ]:
# Select the frozen seed-42 gated model; strict checkpoint loading.
model=HyperKTGated(N_SKILLS,H,d=128,layers=2,heads=8,drop=.1).to(DEVICE)
checkpoint_bytes=z9.read(member(z9,'HyperKT_gated_seed42/best.pt'))
checkpoint_sha256=hashlib.sha256(checkpoint_bytes).hexdigest()
ckpt=torch.load(io.BytesIO(checkpoint_bytes),map_location='cpu',weights_only=False)
print('Checkpoint keys:',list(ckpt) if isinstance(ckpt,dict) else type(ckpt))
assert ckpt.get('stage2_sha256') == hashlib.sha256(Path(next(p for p,z in zs.items() if z is z2)).read_bytes()).hexdigest(), 'Stage 2 ZIP differs from checkpoint training input'
assert ckpt.get('stage3b_sha256') == hashlib.sha256(Path(next(p for p,z in zs.items() if z is z3)).read_bytes()).hexdigest(), 'Stage 3B ZIP differs from checkpoint training input'
state=ckpt.get('model',ckpt.get('model_state_dict',ckpt.get('state_dict',ckpt))) if isinstance(ckpt,dict) else ckpt
model.load_state_dict(state,strict=True)
model.eval()
print('Strict checkpoint load successful; SHA256:',checkpoint_sha256)


## Export causal predictions and event coordinates
The model takes history skill/correctness and target skill as inputs, **not target correctness**. The exported values are observational correctness predictions, not RL policy rewards.

In [ ]:
def export_split(split,d,batch_size=32):
    rows=[];n=len(d['mask']);T=d['mask'].shape[1]
    with torch.inference_mode():
        for start in range(0,n,batch_size):
            end=min(start+batch_size,n)
            hs=torch.as_tensor(d['history_skill'][start:end],dtype=torch.long,device=DEVICE)
            hc=torch.as_tensor(d['history_correct'][start:end],dtype=torch.long,device=DEVICE)
            ts=torch.as_tensor(d['target_skill'][start:end],dtype=torch.long,device=DEVICE)
            probs=torch.sigmoid(model(hs,hc,ts)).cpu().numpy()
            mask=d['mask'][start:end].astype(bool)
            ii,jj=np.where(mask)
            uid=np.asarray(d['user_id'][start:end]).astype(str)
            if uid.ndim!=1: raise ValueError('Expected one user_id per sequence window')
            frame=pd.DataFrame({'split':split,'window_index':ii+start,'position':jj,
                'user_id':uid[ii], 'target_skill':d['target_skill'][start:end][mask].astype(int),
                'target_correct':d['target_correct'][start:end][mask].astype(int),
                'hyperkt_pred_correct':probs[mask].astype(np.float32)})
            rows.append(frame)
    result=pd.concat(rows,ignore_index=True)
    assert len(result)==int(d['mask'].sum())
    assert result['hyperkt_pred_correct'].between(0,1).all()
    assert not result.duplicated(['split','window_index','position']).any()
    path=OUT/f'hyperkt_states_{split}.csv';result.to_csv(path,index=False)
    print(split,len(result),'saved',path)
    return result
exports={s:export_split(s,d) for s,d in [('train',train),('val',val),('test',test)]}


## Audit split isolation and write reproducibility manifest

In [ ]:
users={s:set(f.user_id) for s,f in exports.items()}
assert not (users['train']&users['val'] or users['train']&users['test'] or users['val']&users['test']), 'Learner split overlap'
report={'stage':'objective02_stage2','model':'HyperKT_gated','seed':42,'checkpoint_sha256':checkpoint_sha256,
 'source':'Objective01 Stage2 sequences and Stage3B hypergraph','runtime':str(DEVICE),
 'event_key':['split','window_index','position'],
 'caution':'Window-local coordinates are NOT original ASSISTments event identifiers. Verify original event mapping before joining Stage1 transitions.',
 'no_test_policy_training':True,
 'counts':{s:{'rows':len(f),'users':len(users[s])} for s,f in exports.items()},
 'stage2_manifest':manifest}
(OUT/'stage2_export_audit.json').write_text(json.dumps(report,indent=2,default=str))
print(json.dumps({k:v for k,v in report.items() if k!='stage2_manifest'},indent=2))


## Download results ZIP
Save the archive and upload it here. We will verify the original interaction-event mapping before integrating these outputs into Objective 02 Stage 1.

In [ ]:
out=Path('/content/objective02_stage2_hyperkt_states.zip')
with zipfile.ZipFile(out,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT.glob('hyperkt_states_*.csv'):z.write(p,p.name)
    z.write(OUT/'stage2_export_audit.json','stage2_export_audit.json')
print(out,out.stat().st_size,'bytes')
files.download(str(out))